# PBMC 68K · 10x PBMC 68K

This single-cell RNA-seq notebook uses the dataset-specific marker panel in the cells below. Prepare the inputs listed in [the data guide](../../data/README.md). Start with a fresh kernel and run the annotation cells in order. The last section contains optional comparison methods. The notebook writes its files under `results/runs/single_cell/pbmc68k/`.


In [ ]:
from pathlib import Path
import os

# Both a kernel started alongside the notebook and one started at the repository root work.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "data" / "datasets.tsv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a checkout containing data/datasets.tsv.")
RAW = ROOT / "data" / "raw"
PREPARED = ROOT / "data" / "prepared"
RUN_DIR = ROOT / "results" / "runs" / "single_cell" / "pbmc68k"
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_DIR)
RUN_COMPARISONS = False     # Enable after installing the optional comparison tools.
USE_EXTERNAL_PBMC_LABELS = False
USE_LEGACY_LABEL_MAP = False


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
mtx_dir = RAW / "pbmc68k_10x" / "filtered_matrices_mex" / "hg19"
adata = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", cache=False)
adata.var_names_make_unique()
adata.obs_names_make_unique()

# Join author annotations by 10x barcode when present. Legacy, single-column
# files must follow the exact original barcodes.tsv order.
csv_path = mtx_dir / "pbmc_annot.csv"
df = pd.read_csv(csv_path, dtype=str)
if "pbmcannot" not in df.columns:
    raise ValueError("pbmc_annot.csv must contain the authors' pbmcannot column.")
if "barcode" in df.columns:
    ordered = df.set_index("barcode", verify_integrity=True).reindex(adata.obs_names)
    if ordered["pbmcannot"].isna().any():
        raise ValueError("Some matrix barcodes lack a PBMC reference label.")
elif len(df) == adata.n_obs:
    ordered = df
else:
    raise ValueError("Legacy labels must have one row for every matrix barcode.")
adata.obs["pbmcannot"] = pd.Categorical(ordered["pbmcannot"].to_numpy())
print("Attached columns to adata.obs:", list(df.columns))
#print(adata.obs[list(df.columns)].head())
adata.var_names_make_unique()   # avoids duplicate symbol problems in downstream tools
adata.obs_names_make_unique()
print(f"START  | cells={adata.n_obs:,}  genes={adata.n_vars:,}")
adata.var_names = adata.var_names.astype(str)
adata.var_names_make_unique()
vn_up = adata.var_names.str.upper()

# --- Define QC gene sets and SAVE as columns in adata.var ---
# Human and mouse 'mt-' both become 'MT-' after uppercasing
adata.var["mt"]   = vn_up.str.startswith("MT-")
adata.var["ribo"] = vn_up.str.startswith(("RPS","RPL","MRPS","MRPL"))
adata.var["hb"]   = adata.var_names.str.match(r"^(HB[ABEDM][A-Z0-9]*)", case=False)

# --- Compute QC metrics using those var columns ---
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt","ribo","hb"],   # <- names of columns in adata.var
    percent_top=None,
    log1p=False,
    inplace=True
)

print(adata.obs[["n_genes_by_counts","total_counts","pct_counts_mt"]].describe())

# --- Choose thresholds (data-driven) ---
min_genes = 200
min_cells_gene = 3
umi_hi   = np.percentile(adata.obs["total_counts"], 99.5)
genes_hi = np.percentile(adata.obs["n_genes_by_counts"], 99.5)
mito_hi  = max(20.0, np.percentile(adata.obs["pct_counts_mt"], 95))

print(f"Thresholds → min_genes={min_genes}, umi_hi≈{umi_hi:.0f}, genes_hi≈{genes_hi:.0f}, mito_hi≈{mito_hi:.1f}%")

keep_cells = (
    (adata.obs["n_genes_by_counts"] >= min_genes) &
    (adata.obs["n_genes_by_counts"] <= genes_hi) &
    (adata.obs["total_counts"]      <= umi_hi) &
    (adata.obs["pct_counts_mt"]     <= mito_hi)
)
before = adata.n_obs
adata = adata[keep_cells].copy()
print(f"CELL FILTER | kept {adata.n_obs:,}/{before:,} ({adata.n_obs/before:.1%})")

# --- Filter low-prevalence genes ---
before_g = adata.n_vars
sc.pp.filter_genes(adata, min_cells=min_cells_gene)
print(f"GENE FILTER | kept {adata.n_vars:,}/{before_g:,} (≥{min_cells_gene} cells)")

# --- Normalize & log1p ---
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
print(f"READY | cells={adata.n_obs:,} genes={adata.n_vars:,}")

# Rename the annotation column
adata.obs = adata.obs.rename(columns={"pbmcannot": "pseudo_cell_type"})

# --- PBMC marker sets ---
SKIN_MARKERS = {
    # -------------------------------------------------
    # CD8+/CD45RA+ Naive Cytotoxic (naive CD8 T)
    # -------------------------------------------------
    "CD8+/CD45RA+ Naive Cytotoxic": [
        "CD3D","CD3E","TRAC",
        "CD8A","CD8B",
        "CCR7","LEF1","TCF7",
        "LTB","IL7R",
        "MAL","LST1"   # (optional; remove LST1 if you see myeloid leakage)
    ],

    # -------------------------------------------------
    # CD4+/CD25 T Reg (Treg)
    # -------------------------------------------------
    "CD4+/CD25 T Reg": [
        "CD3D","CD3E","TRAC",
        "CD4",
        "IL2RA","FOXP3","IKZF2",
        "CTLA4","TIGIT","TNFRSF18",
        "CCR7","LTB"
    ],

    # -------------------------------------------------
    # CD8+ Cytotoxic T
    # -------------------------------------------------
    "CD8+ Cytotoxic T": [
        "CD3D","CD3E","TRAC",
        "CD8A","CD8B",
        "NKG7","GNLY",
        "GZMB","GZMH","PRF1",
        "CTSW","KLRD1",
        "CCL5"
    ],

    # -------------------------------------------------
    # CD56+ NK
    # -------------------------------------------------
    "CD56+ NK": [
        "NKG7","GNLY","PRF1",
        "GZMB","GZMH","CTSW",
        "KLRD1","FCGR3A",
        "TRDC",        # optional (rare)
        "XCL1","XCL2"
    ],

    # -------------------------------------------------
    # CD19+ B
    # -------------------------------------------------
    "CD19+ B": [
        "MS4A1","CD79A","CD79B",
        "CD74","HLA-DRA","HLA-DRB1",
        "CD37","CD19",
        "BANK1","CD22",
        "CD83"  # activation sometimes
    ],

    # -------------------------------------------------
    # CD14+ Monocyte (classical monocytes)
    # -------------------------------------------------
    "CD14+ Monocyte": [
        "LYZ","S100A8","S100A9",
        "CTSS","FCN1","LGALS3",
        "LST1","TYROBP","FCER1G",
        "CTSD","MNDA",
        "IL1B"
    ],

    # -------------------------------------------------
    # CD4+/CD45RO+ Memory
    # -------------------------------------------------
    "CD4+/CD45RO+ Memory": [
        "CD3D","CD3E","TRAC",
        "CD4",
        "IL7R","LTB",
        "CCR7",     # central memory
        "MAL","NOSIP",
        "TCF7","LEF1",   # may be lower than naive
        "CXCR4"
    ],

    # -------------------------------------------------
    # CD4+/CD45RA+/CD25- Naive T (naive CD4)
    # -------------------------------------------------
    "CD4+/CD45RA+/CD25- Naive T": [
        "CD3D","CD3E","TRAC",
        "CD4",
        "CCR7","LEF1","TCF7",
        "IL7R","LTB",
        "MAL","NOSIP",
        "LST1"  # optional; remove if contaminating monocytes
    ],

    # -------------------------------------------------
    # Dendritic (cDC / pDC mixed depending on dataset)
    # -------------------------------------------------
    "Dendritic": [
        "FCER1A","CD1C","CLEC10A",   # cDC2
        "ITGAX","LILRA4",            # ITGAX general DC; LILRA4 pDC
        "GZMB",                      # pDC hallmark (often)
        "HLA-DRA","HLA-DRB1",
        "IRF7"
    ],

    # -------------------------------------------------
    # CD34+ (HSPC / progenitors)
    # -------------------------------------------------
    "CD34+": [
        "CD34","SPINK2","GATA2",
        "MPO",       # may indicate myeloid progenitors
        "HBB",       # remove if RBC contamination
        "TYMP",
        "MEIS1",
        "AVP"        # optional depending on platform
    ],

    # -------------------------------------------------
    # CD4+ T Helper2 (rare; your dataset has 19 cells)
    # -------------------------------------------------
    "CD4+ T Helper2": [
        "CD3D","CD3E","TRAC",
        "CD4",
        "IL7R",
        "GATA3","IL4","IL5","IL13",
        "CCR4","CCR6",
        "ICOS"
    ],
}






In [ ]:
#Method1 (Fast method without Dirichlet, marked intial supervised cell using adaptive density distribution of each marker set)
import scanpy as sc
import numpy as np
import pandas as pd

# Load your human cell marker (total 78 markers) file
#df = pd.read_csv("skin_marker.csv", encoding="latin1")

# Clean column names (just in case)
#df.columns = [c.strip().lower() for c in df.columns]

# Group markers by cell type
#PBMC_MARKERS = (
#    df.groupby("cell_name")["marker"]
#      .apply(lambda x: sorted(set(x.str.upper().str.strip())))
#      .to_dict()
#)



def adaptive_marker_threshold(adata, gene, quantile=0.75):
    vals = adata[:, gene].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.asarray(vals).ravel()
    vals = vals[vals > 0]
    if vals.size == 0:
        return np.inf
    return float(np.quantile(vals, quantile))

def weak_label_quota_with_min_cells(
    adata,
    marker_sets,
    gene_quantile=0.75,
    target_frac=0.25,
    min_cells_per_type=200,
    min_score=0.2,
    score_mode="frac_high",   # "frac_high" as you used; keep for clarity
    verbose=True
):
    """
    Makes weak labels with:
      - total labeled fraction ~ target_frac
      - every labeled type has >= min_cells_per_type
      - optional min_score threshold
    """

    N = adata.n_obs
    target_total = int(round(target_frac * N))

    # --- 1) Compute per-type marker score (frac of markers above gene-specific threshold)
    scores = {}
    for cell_type, genes in marker_sets.items():
        genes = [g for g in genes if g in adata.var_names]
        if len(genes) == 0:
            continue
        thr = {g: adaptive_marker_threshold(adata, g, quantile=gene_quantile) for g in genes}

        M = sc.get.obs_df(adata, keys=list(thr.keys()))
        expr_high = M.gt(pd.Series(thr))
        frac_high = expr_high.sum(axis=1) / len(thr)

        scores[cell_type] = frac_high.astype(float)

    if len(scores) == 0:
        raise ValueError("No marker sets had genes present in adata.var_names")

    score_df = pd.DataFrame(scores, index=adata.obs_names)

    # --- 2) Best type + best score per cell
    best_score = score_df.max(axis=1).astype(float)
    best_type  = score_df.idxmax(axis=1).astype(str)

    # --- 3) Eligible cells by min_score
    eligible_mask = (best_score >= float(min_score))
    if verbose:
        print(f"[quota] Eligible (score >= {min_score}): {eligible_mask.sum()}/{N} ({eligible_mask.mean():.2%})")

    # --- 4) Keep only types that have >= min_cells_per_type eligible cells
    eligible_types_counts = best_type[eligible_mask].value_counts()
    keep_types = eligible_types_counts[eligible_types_counts >= min_cells_per_type].index.tolist()

    if verbose:
        print(f"[quota] Types with >= {min_cells_per_type} eligible cells: {len(keep_types)}/{score_df.shape[1]}")

    if len(keep_types) == 0:
        # nobody qualifies; everything unlabeled
        adata.obs["weak_label"] = pd.Categorical(["unlabeled"] * N)
        adata.obs["weak_conf"]  = best_score.values
        return adata, score_df

    # If too many types * min_cells exceeds target_total, we must drop some types
    # Keep the most abundant eligible types first.
    max_types_possible = target_total // min_cells_per_type
    if max_types_possible == 0:
        if verbose:
            print("[quota][warn] target_frac too small for min_cells_per_type. Labeling none.")
        adata.obs["weak_label"] = pd.Categorical(["unlabeled"] * N)
        adata.obs["weak_conf"]  = best_score.values
        return adata, score_df

    if len(keep_types) > max_types_possible:
        keep_types = eligible_types_counts.loc[keep_types].sort_values(ascending=False).index[:max_types_possible].tolist()
        if verbose:
            print(f"[quota][warn] Too many types for target budget; keeping top {len(keep_types)} types by eligible size.")

    # --- 5) For kept types, decide how many cells to label per type
    # Base allocation: min_cells_per_type each
    base = {t: min_cells_per_type for t in keep_types}
    base_total = sum(base.values())

    # Remaining budget to reach target_total
    remaining = max(0, target_total - base_total)

    # Available eligible cells per kept type
    avail = eligible_types_counts.loc[keep_types].to_dict()

    # Extra allocation proportional to availability beyond base
    extra = {t: 0 for t in keep_types}
    if remaining > 0:
        weights = np.array([max(0, avail[t] - base[t]) for t in keep_types], dtype=float)
        if weights.sum() > 0:
            props = weights / weights.sum()
            raw_extra = np.floor(props * remaining).astype(int)
            for t, e in zip(keep_types, raw_extra):
                extra[t] = int(e)

            # distribute leftover one-by-one
            leftover = remaining - sum(extra.values())
            if leftover > 0:
                order = np.argsort(-props)  # highest proportion first
                k = 0
                while leftover > 0:
                    t = keep_types[order[k % len(order)]]
                    extra[t] += 1
                    leftover -= 1
                    k += 1

    # Final per-type quota, capped by available eligible cells
    quota = {}
    for t in keep_types:
        q = base[t] + extra[t]
        q = min(q, avail[t])  # cannot exceed eligible availability
        quota[t] = int(q)

    # If capping reduced total, we can optionally refill from types with spare capacity
    chosen_total = sum(quota.values())
    deficit = target_total - chosen_total
    if deficit > 0:
        # refill greedily from types with spare eligible cells
        spare = {t: avail[t] - quota[t] for t in keep_types}
        order = sorted(keep_types, key=lambda t: spare[t], reverse=True)
        i = 0
        while deficit > 0 and i < len(order):
            t = order[i]
            take = min(deficit, spare[t])
            if take > 0:
                quota[t] += int(take)
                deficit -= int(take)
            i += 1

    # --- 6) Select top-scoring cells within each kept type according to quota
    weak_label = np.array(["unlabeled"] * N, dtype=object)

    # Precompute indices per type among eligible cells
    best_type_arr = best_type.values
    best_score_arr = best_score.values
    eligible_idx = np.where(eligible_mask.values)[0]

    for t in keep_types:
        idx_t = eligible_idx[best_type_arr[eligible_idx] == t]
        if idx_t.size == 0:
            continue
        # sort by best_score descending
        idx_sorted = idx_t[np.argsort(-best_score_arr[idx_t])]
        pick = idx_sorted[:quota[t]]
        weak_label[pick] = t

    adata.obs["weak_label"] = pd.Categorical(weak_label)
    adata.obs["weak_conf"]  = best_score.values

    if verbose:
        vc = adata.obs["weak_label"].value_counts()
        labeled = (adata.obs["weak_label"] != "unlabeled").sum()
        print(f"[quota] Target={target_frac:.0%} | Achieved={labeled/N:.2%} | labeled={labeled}/{N}")
        print(vc)
        small = vc[(vc.index != "unlabeled") & (vc < min_cells_per_type)]
        if len(small) > 0:
            print("[quota][ERROR] still found small labeled types:", small.to_dict())
        else:
            print(f"[quota] OK: all labeled types have >= {min_cells_per_type} cells")

    return adata, score_df
adata, marker_score_df = weak_label_quota_with_min_cells(
    adata,
    SKIN_MARKERS,
    gene_quantile=0.75,
    target_frac=0.6,        # 20–30% desired
    min_cells_per_type=10,  # enforce minimum cluster size
    min_score=0.2,
    verbose=True
)





In [ ]:
#Method3

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

def row_softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / (E.sum(axis=1, keepdims=True) + 1e-12)

def gcn_seed_labeling_quota(
    adata,
    score_df,                         # cells x types marker_score_df
    target_frac=0.40,                 # desired final labeled fraction
    temperature=0.25,                 # 0.15–0.5 (lower = sharper)
    alpha=0.65,                       # 0.5–0.8 (lower = less oversmoothing)
    n_iter=50,
    tol=1e-4,                         # early stop
    # adaptive gates (quantiles)
    q_score=0.40,                     # gate on top1 >= quantile(top1)
    q_margin=0.35,                    # gate on margin >= quantile(margin)
    # quota controls
    min_cells_per_type=100,           # ensure each type has >= this many seeds (if possible)
    # minimum-size filter after quota (keep rare types!)
    min_cells_per_type_pct=0.001,     # 0.1% of n (≈68 for 68k)
    min_cells_floor=30,               # absolute min
    # graph construction
    neighbors_k=30,
    pca_comps=50,
    use_rep="X_pca",
    label_col="weak_label_gcn",
    conf_col="seed_conf_gcn",
    verbose=True
):
    # ---------- 0) Align score_df ----------
    if not isinstance(score_df, pd.DataFrame):
        raise ValueError("score_df must be a pandas DataFrame (cells x types).")

    score_df = score_df.copy()
    score_df.index = score_df.index.astype(str)
    score_df = score_df.reindex(adata.obs_names.astype(str))
    score_df = score_df.loc[:, score_df.notna().any(axis=0)]
    if score_df.shape[1] == 0:
        raise ValueError("score_df has no usable type columns after filtering.")

    types = score_df.columns.astype(str).tolist()
    S0 = score_df.fillna(0.0).to_numpy(dtype=float)

    # ---------- 1) Build sharpened probability seeds ----------
    # softmax produces probability-like vectors; temperature sharpens
    Y0 = row_softmax(S0 / float(temperature))

    # ---------- 2) Ensure PCA + neighbors ----------
    if (use_rep not in adata.obsm) or (adata.obsm[use_rep] is None):
        sc.pp.pca(adata, n_comps=pca_comps, svd_solver="arpack")
        use_rep = "X_pca"

    if "connectivities" not in adata.obsp:
        sc.pp.neighbors(
            adata,
            n_neighbors=neighbors_k,
            n_pcs=min(pca_comps, adata.obsm[use_rep].shape[1])
        )

    A = adata.obsp["connectivities"]
    if not sparse.isspmatrix(A):
        A = sparse.csr_matrix(A)

    N = adata.n_obs
    I = sparse.eye(N, format="csr")
    A_tilde = (A + I).tocsr()

    deg = np.asarray(A_tilde.sum(axis=1)).ravel()
    deg_inv_sqrt = 1.0 / np.sqrt(deg + 1e-12)
    D_inv_sqrt = sparse.diags(deg_inv_sqrt, format="csr")
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

    # ---------- 3) Propagate with early stop ----------
    Y = Y0.copy()
    for it in range(int(n_iter)):
        Y_new = alpha * (A_hat @ Y) + (1.0 - alpha) * Y0
        Y_new = np.asarray(Y_new)
        diff = float(np.mean(np.abs(Y_new - Y)))
        Y = Y_new
        if diff < tol:
            if verbose:
                print(f"[GCN] early stop at iter={it+1}, mean|Δ|={diff:.2e}")
            break

    # ---------- 4) Compute top1/top2/margin ----------
    top1_idx = Y.argmax(axis=1)
    top1 = Y[np.arange(N), top1_idx]

    Y2 = Y.copy()
    Y2[np.arange(N), top1_idx] = -np.inf
    top2 = Y2.max(axis=1)
    margin = top1 - top2

    # ---------- 5) Adaptive gate (key fix vs fixed delta=0.15) ----------
    min_score_eff = float(np.quantile(top1, q_score))
    delta_eff = float(np.quantile(margin, q_margin))

    gate = (top1 >= min_score_eff) & (margin >= delta_eff)

    if verbose:
        print(f"[GCN] adaptive gate: min_score_eff={min_score_eff:.3f}, "
              f"delta_eff={delta_eff:.3f}, pass={gate.mean():.2%}")
        print(f"[GCN] median top1={np.median(top1):.3f}, median margin={np.median(margin):.3f}")

    # preliminary labels + conf
    pred_type = np.array(types, dtype=object)[top1_idx]
    conf = top1.copy()

    # ---------- 6) QUOTA selection to reach target_frac and keep many types ----------
    target_n = int(round(float(target_frac) * N))
    idx_gate = np.where(gate)[0]

    labels = np.array(["unlabeled"] * N, dtype=object)
    seed_conf = np.zeros(N, dtype=float)

    if idx_gate.size == 0 or target_n == 0:
        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col] = seed_conf
        return {"Y_propagated": Y, "top1": top1, "top2": top2, "margin": margin, "types": types}

    # group gated indices by predicted type
    gate_df = pd.DataFrame({
        "idx": idx_gate,
        "type": pred_type[idx_gate],
        "conf": conf[idx_gate]
    })

    # ensure each type gets min_cells_per_type if possible
    picked = []
    per_type_min = int(min_cells_per_type)

    for t in types:
        sub = gate_df[gate_df["type"] == t].sort_values("conf", ascending=False)
        if sub.shape[0] == 0:
            continue
        take = min(per_type_min, sub.shape[0])
        picked.extend(sub["idx"].values[:take].tolist())

    picked = list(dict.fromkeys(picked))  # unique preserve order
    remaining_budget = max(0, target_n - len(picked))

    if remaining_budget > 0:
        # fill remaining with highest confidence gated cells not yet picked
        rest = gate_df[~gate_df["idx"].isin(picked)].sort_values("conf", ascending=False)
        picked.extend(rest["idx"].values[:remaining_budget].tolist())

    # assign picked
    labels[picked] = pred_type[picked]
    seed_conf[picked] = conf[picked]

    # ---------- 7) Drop very small types (looser than your pct=0.005) ----------
    min_cells_pct = int(np.floor(float(min_cells_per_type_pct) * N))
    min_cells = max(int(min_cells_floor), int(min_cells_pct), 5)

    vc = pd.Series(labels).value_counts()
    keep_types = set(vc[vc >= min_cells].index.tolist())
    keep_types.discard("unlabeled")

    small_mask = (labels != "unlabeled") & (~pd.Series(labels).isin(keep_types).values)
    labels[small_mask] = "unlabeled"
    seed_conf[small_mask] = 0.0

    # ---------- 8) Write back ----------
    adata.obs[label_col] = pd.Categorical(labels)
    if "unlabeled" not in adata.obs[label_col].cat.categories:
        adata.obs[label_col] = adata.obs[label_col].cat.add_categories(["unlabeled"])
    adata.obs[label_col] = adata.obs[label_col].fillna("unlabeled")
    adata.obs[conf_col] = seed_conf

    if verbose:
        print(f"[GCN] min_cells_per_type used={min_cells}")
        print("[GCN] final label distribution:")
        print(adata.obs[label_col].value_counts())
        print(f"[GCN] labeled fraction={(adata.obs[label_col] != 'unlabeled').mean():.2%}")

    return {"Y_propagated": Y, "top1": top1, "top2": top2, "margin": margin, "types": types}



diag = gcn_seed_labeling_quota(
    adata,
    score_df=marker_score_df,

    # labeling quota
    target_frac=0.6,        # was 0.4 → allow more labeled cells

    # propagation
    temperature=0.1,        # was 0.1 → softer probability distribution
    alpha=0.5,              # was 0.5 → stronger graph propagation
    n_iter=100,              # was 50 → more smoothing

    # graph construction
    neighbors_k=50,          # was 50 → smoother graph
    pca_comps=50,

    label_col="weak_label",
    conf_col="seed_conf",

    verbose=True
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import StandardScaler

def dp_seed_by_marker_sets_soft_no_pca(
    adata,
    marker_sets,
    n_components=30,
    weight_concentration_prior=0.05,
    random_state=0,
    per_gene_pos_quantile=0.70,
    cluster_score_min=0.20,
    min_cells_cluster=100,
    label_col_prefix="dp_seed",
    conf_col_prefix="dp_seed_conf",
    verbose=True
):
    diagnostics = {}
    all_vars = set(adata.var_names.astype(str))
    n_cells = adata.n_obs

    # --- Pre-filter thresholds ---
    min_genes_present     = 3
    min_expressed_markers = 2
    min_cells_per_gene    = 20
    min_cell_enrichment   = 0.05

    for setname, genes in marker_sets.items():
        # 0) Keep only markers present in adata
        genes = [g for g in genes if g in all_vars]
        if len(genes) < min_genes_present:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {len(genes)} markers present in adata.")
            continue

        # 1) Extract expression
        Xg = adata[:, genes].X
        if hasattr(Xg, "toarray"):
            Xg = Xg.toarray()
        Xg = np.asarray(Xg, dtype=float)

        # 2) Pre-filter 1: expressed markers
        expr_per_gene = (Xg > 0).sum(axis=0)
        expressed_markers = int(np.sum(expr_per_gene >= min_cells_per_gene))
        if expressed_markers < min_expressed_markers:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {expressed_markers} expressed markers.")
            continue

        # 3) Pre-filter 2: cell enrichment
        cells_with_any_marker = (Xg > 0).sum(axis=1) > 0
        cell_enrichment = float(np.mean(cells_with_any_marker))
        if cell_enrichment < min_cell_enrichment:
            if verbose:
                print(f"[DP-soft][skip] {setname}: cell enrichment {cell_enrichment:.4f} < {min_cell_enrichment}.")
            continue

        # 4) Per-gene thresholds
        gene_thr = {}
        for j, g in enumerate(genes):
            v = Xg[:, j]
            pos = v[v > 0]
            if pos.size >= 20:
                gene_thr[g] = float(np.quantile(pos, per_gene_pos_quantile))
            elif pos.size > 0:
                gene_thr[g] = float(np.median(pos))
            else:
                gene_thr[g] = np.inf

        # 5) Cell marker scores
        high_mask = np.zeros_like(Xg, dtype=bool)
        for j, g in enumerate(genes):
            thr = gene_thr[g]
            if np.isinf(thr):
                continue
            high_mask[:, j] = Xg[:, j] > thr

        marker_score = high_mask.sum(axis=1) / max(1, len(genes))

        # 6) Standardize + BGM (NO PCA)
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xg_scaled = scaler.fit_transform(Xg)

        bgm = BayesianGaussianMixture(
            n_components=n_components,
            weight_concentration_prior_type="dirichlet_process",
            weight_concentration_prior=weight_concentration_prior,
            covariance_type="full",
            max_iter=1000,
            n_init=1,
            random_state=random_state,
        )
        bgm.fit(Xg_scaled)
        cl = bgm.predict(Xg_scaled)

        # 7) Cluster-level stats
        df_cluster = pd.DataFrame({"cluster": cl, "score": marker_score})
        cluster_summary = df_cluster.groupby("cluster")["score"].agg(["mean", "count"])

        good_clusters = cluster_summary[
            (cluster_summary["mean"] >= cluster_score_min) &
            (cluster_summary["count"] >= min_cells_cluster)
        ].index.tolist()

        # 8) Assign labels + confidence
        label_col = f"{label_col_prefix}_{setname}"
        conf_col  = f"{conf_col_prefix}_{setname}"

        labels = np.array(["unlabeled"] * n_cells, dtype=object)
        confs  = np.zeros(n_cells, dtype=float)

        if len(good_clusters) > 0:
            max_score = marker_score.max() if marker_score.max() > 0 else 1.0
            for cid in good_clusters:
                idx = np.where(cl == cid)[0]
                labels[idx] = setname
                confs[idx] = marker_score[idx] / max_score

        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col]  = confs

        # 9) Diagnostics
        n_lab = (labels != "unlabeled").sum()
        pct_lab = n_lab / n_cells * 100.0
        diagnostics[setname] = {
            "n_markers_used": len(genes),
            "n_clusters": int(len(cluster_summary)),
            "good_clusters": good_clusters,
            "n_labelled": int(n_lab),
            "pct_labelled": float(pct_lab),
            "cluster_summary": cluster_summary,
            "expressed_markers": expressed_markers,
            "cell_enrichment": cell_enrichment,
            "n_features_used": int(Xg_scaled.shape[1]),
        }

        if verbose:
            print(
                f"{setname}: labelled {n_lab}/{n_cells} ({pct_lab:.2f}%), "
                f"good_clusters={len(good_clusters)}, n_features={Xg_scaled.shape[1]}"
            )

    # --- Build consensus weak label across marker sets ---
    label_cols = [f"{label_col_prefix}_{s}" for s in marker_sets.keys() if f"{label_col_prefix}_{s}" in adata.obs]
    conf_cols  = [f"{conf_col_prefix}_{s}" for s in marker_sets.keys() if f"{conf_col_prefix}_{s}" in adata.obs]

    if len(label_cols) > 0:
        lab_df = adata.obs[label_cols].astype(str)
        conf_df = adata.obs[conf_cols].astype(float).fillna(0.0)

        dp_consensus_label = []
        dp_consensus_conf  = []

        for i in adata.obs_names:
            row_labels = lab_df.loc[i].values
            row_confs  = conf_df.loc[i].values
            row_confs = np.where(row_labels == "unlabeled", 0.0, row_confs)

            best_idx = int(np.argmax(row_confs))
            best_conf = float(row_confs[best_idx])
            best_label = row_labels[best_idx] if best_conf > 0 else "unlabeled"

            dp_consensus_label.append(best_label)
            dp_consensus_conf.append(best_conf)

        seed_labels = pd.Categorical(dp_consensus_label)
        if "unlabeled" not in seed_labels.categories:
            seed_labels = seed_labels.add_categories(["unlabeled"])
        adata.obs["weak_label"] = seed_labels
        adata.obs["weak_conf"]  = np.array(dp_consensus_conf, dtype=float)

        # --- Final size filter ---
        threshold = max(100, int(0.002 * adata.n_obs))
        counts = adata.obs["weak_label"].value_counts()

        keep_types = set(counts[counts >= threshold].index.tolist())

        mask_keep = adata.obs["weak_label"].isin(keep_types)
        adata.obs.loc[~mask_keep, "weak_label"] = "unlabeled"
        adata.obs.loc[~mask_keep, "weak_conf"]  = 0.0

        if verbose:
            print("\n[DP-soft no PCA] Final weak_label distribution:")
            print(adata.obs["weak_label"].value_counts(normalize=True) * 100)

    return diagnostics

In [ ]:
N = adata.n_obs
n_components = min(100, int(np.sqrt(N))) # capped adaptive choice

diag = dp_seed_by_marker_sets_soft_no_pca(
    adata,
    SKIN_MARKERS,                    # PCA dimension for each marker block
    per_gene_pos_quantile=0.4,
    cluster_score_min=0.1,
    min_cells_cluster=100,
    n_components=n_components,
    weight_concentration_prior=0.05,
    random_state=42
)

for k, v in diag.items():
    print(f"{k}: labelled {v['n_labelled']} cells ({v['pct_labelled']:.2f}%)")

In [ ]:
# PCA
sc.pp.pca(adata, n_comps=30, svd_solver="arpack")

# Graph (only if using Leiden / GCN)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
def get_training_data(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.3,
    max_pcs=30
):
    X = np.asarray(adata.obsm[X_key], dtype=np.float32)
    if X.ndim != 2:
        raise ValueError(f"{X_key} must be 2D.")

    # use only first max_pcs PCs
    X = X[:, :min(max_pcs, X.shape[1])]

    y = adata.obs[seed_col].astype(str).values

    if conf_col in adata.obs.columns:
        conf = np.asarray(adata.obs[conf_col], dtype=float)
        known_mask = (y != "unlabeled") & (conf >= min_seed_conf)
    else:
        known_mask = (y != "unlabeled")

    if known_mask.sum() < 20:
        raise ValueError(f"Too few confident labeled seeds: {known_mask.sum()}")

    return X, y, known_mask


from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def rbf_svm_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_svm_rbf",
    out_conf_col="svm_rbf_conf",
    C=2.0,
    gamma="scale",
    min_seed_conf=0.3,
    min_conf=0.6,
    max_pcs=30
):
    X, y, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        SVC(
            C=C,
            gamma=gamma,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        )
    )

    clf.fit(X[known_mask], y[known_mask])

    proba = clf.predict_proba(X)
    pred = clf.classes_[np.argmax(proba, axis=1)]
    conf = np.max(proba, axis=1)

    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return clf

clf_rbf = rbf_svm_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.30,
    min_conf=0.60,
    max_pcs=30
)

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from collections import Counter

# ----- Feature space -----
X = np.asarray(adata.obsm["X_pca"][:, :30], dtype=float)

# ----- Seed labels -----
labels_seed = adata.obs["weak_label"].astype(str).values
conf = adata.obs.get("weak_conf", pd.Series(np.ones(len(labels_seed))))

# Use only confident seeds
seed_mask = (labels_seed != "unlabeled") & (conf > 0.3)

known_types = np.unique(labels_seed[seed_mask])
print("Known types:", known_types)

# ----- Fit KMeans -----
k = min(30, max(10, int(np.sqrt(adata.n_obs/2))))   # adaptive cluster number
km = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20,
    max_iter=500
)
km.fit(X)

# ----- Compute centroids of seed types (fallback) -----
centroids = []
for t in known_types:
    centroids.append(X[seed_mask & (labels_seed == t)].mean(axis=0))
C = np.vstack(centroids)

# ----- Map clusters to labels -----
cluster_to_type = {}
cluster_conf = {}

for c in np.unique(km.labels_):

    idx = np.where(km.labels_ == c)[0]

    seed_in_c = labels_seed[idx]
    seed_in_c = seed_in_c[seed_in_c != "unlabeled"]

    # Case 1: cluster has seeds
    if len(seed_in_c) > 0:

        counts = Counter(seed_in_c)
        maj_label, maj_count = counts.most_common(1)[0]

        cluster_to_type[c] = maj_label
        cluster_conf[c] = maj_count / len(idx)

    # Case 2: no seeds → assign by nearest seed centroid
    else:

        center = km.cluster_centers_[c].reshape(1, -1)
        nearest = pairwise_distances(center, C).argmin()

        cluster_to_type[c] = known_types[nearest]
        cluster_conf[c] = 0.0


# ----- Assign labels to cells -----
labels_out = np.array([cluster_to_type[c] for c in km.labels_])
conf_out = np.array([cluster_conf[c] for c in km.labels_])

adata.obs["label_kmeans"] = pd.Categorical(labels_out)
adata.obs["kmeans_conf"] = conf_out

from sklearn.neighbors import KNeighborsClassifier

def knn_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_knn",
    out_conf_col="knn_conf",
    n_neighbors=9,
    min_seed_conf=0.3,
    min_conf=0.55,
    max_pcs=30
):
    X, y, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    knn = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights="distance",
        p=2
    )
    knn.fit(X[known_mask], y[known_mask])

    pred = knn.predict(X)
    proba = knn.predict_proba(X)
    conf = np.max(proba, axis=1)

    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return knn

clf_knn = knn_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    n_neighbors=9,
    min_seed_conf=0.30,
    min_conf=0.55,
    max_pcs=30
)

from sklearn.ensemble import RandomForestClassifier

def rf_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_rf",
    out_conf_col="rf_conf",
    n_estimators=500,
    max_depth=18,
    min_samples_leaf=5,
    min_seed_conf=0.3,
    min_conf=0.55,
    max_pcs=30,
    random_state=42
):
    X, y, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features="sqrt",
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=random_state
    )

    rf.fit(X[known_mask], y[known_mask])

    pred = rf.predict(X)
    proba = rf.predict_proba(X)
    conf = np.max(proba, axis=1)

    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return rf


clf_rf = rf_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.30,
    min_conf=0.55,
    max_pcs=30
)


from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier

def mlp_labeling(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    out_col="label_mlp",
    out_conf_col="mlp_conf",
    hidden_layer_sizes=(128, 64),
    alpha=1e-3,
    max_iter=400,
    min_seed_conf=0.3,
    min_conf=0.60,
    max_pcs=30,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
):
    X, y_str, known_mask = get_training_data(
        adata,
        X_key=X_key,
        seed_col=seed_col,
        conf_col=conf_col,
        min_seed_conf=min_seed_conf,
        max_pcs=max_pcs
    )

    le = LabelEncoder()
    y_known = le.fit_transform(y_str[known_mask])

    if len(le.classes_) < 2:
        raise ValueError(f"Only one labeled class present in seeds: {le.classes_[0]}")

    clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation="relu",
            solver="adam",
            alpha=alpha,
            learning_rate_init=1e-3,
            max_iter=max_iter,
            early_stopping=early_stopping,
            validation_fraction=validation_fraction,
            n_iter_no_change=20,
            random_state=random_state,
            verbose=False
        )
    )

    clf.fit(X[known_mask], y_known)

    proba = clf.predict_proba(X)
    pred_int = np.argmax(proba, axis=1)
    pred_lbl = le.inverse_transform(pred_int)
    conf = np.max(proba, axis=1)

    y_out = pred_lbl.copy()
    y_out[conf < min_conf] = "unlabeled"

    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

    return clf, le


clf_mlp, le = mlp_labeling(
    adata,
    seed_col="weak_label",
    conf_col="weak_conf",
    hidden_layer_sizes=(128, 64),
    alpha=1e-3,
    min_seed_conf=0.30,
    min_conf=0.60,
    max_pcs=30,
    random_state=42
)


# 6) Consensus label for previously unknown cells
methods = np.vstack([
    adata.obs["label_svm_rbf"].astype(str).values,
    adata.obs["label_kmeans"].astype(str).values,
    adata.obs["label_knn"].astype(str).values,
    adata.obs["label_rf"].astype(str).values,
     adata.obs["label_mlp"].astype(str).values
])  # shape: 5 x n_cells

consensus = np.array(["unlabeled"] * adata.n_obs, dtype=object)
agree_frac = np.zeros(adata.n_obs)

for i in range(adata.n_obs):
    votes = [m[i] for m in methods]
    # don’t count 'unlabeled' as a valid vote if other labels exist
    filtered = [v for v in votes if v != "unlabeled"]
    if len(filtered) == 0:
        consensus[i] = "unlabeled"
        agree_frac[i] = 0.0
    else:
        c = Counter(filtered)
        maj_label, maj_votes = c.most_common(1)[0]
        consensus[i] = maj_label
        agree_frac[i] = maj_votes / 5.0

adata.obs["label_consensus"] = pd.Categorical(consensus)
adata.obs["consensus_agree_frac"] = agree_frac



In [ ]:
# Run Leiden clustering
sc.tl.leiden(adata, resolution=1.6, key_added="label_leiden")


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from collections import OrderedDict
from scipy import sparse

def prepare_marker_dict_and_group_order(adata, marker_dict, groupby_col="label_consensus"):
    """
    Make group labels categorical in a smart order:
    marker_dict keys first, then extra labels present in adata.
    Also keep only marker genes present in adata.var_names.
    """
    adata = adata.copy()
    adata.obs[groupby_col] = adata.obs[groupby_col].astype(str)

    labels_present = list(pd.unique(adata.obs[groupby_col]))
    marker_order = list(marker_dict.keys())

    ordered_labels = [x for x in marker_order if x in labels_present]
    extra_labels = [x for x in labels_present if x not in ordered_labels]
    final_order = ordered_labels + extra_labels

    adata.obs[groupby_col] = pd.Categorical(
        adata.obs[groupby_col],
        categories=final_order,
        ordered=True
    )

    marker_dict_present = OrderedDict()
    missing_marker_summary = {}

    for ct, genes in marker_dict.items():
        genes_present = [g for g in genes if g in adata.var_names]
        genes_missing = [g for g in genes if g not in adata.var_names]

        seen = set()
        genes_present = [g for g in genes_present if not (g in seen or seen.add(g))]

        if len(genes_present) > 0:
            marker_dict_present[ct] = genes_present
        if len(genes_missing) > 0:
            missing_marker_summary[ct] = genes_missing

    return adata, marker_dict_present, final_order, missing_marker_summary


def get_top_n_marker_genes_by_cluster_mean(
    adata,
    marker_dict,
    groupby_col="label_consensus",
    top_n=3,
):
    """
    For each cluster/cell type in marker_dict, select the top_n marker genes
    by mean expression within that cluster, restricted to the marker genes
    listed for that cluster.
    """
    adata2, marker_dict_present, final_order, missing_marker_summary = prepare_marker_dict_and_group_order(
        adata, marker_dict, groupby_col=groupby_col
    )

    # flatten unique marker genes
    all_marker_genes = []
    for genes in marker_dict_present.values():
        all_marker_genes.extend(genes)

    seen = set()
    all_marker_genes = [g for g in all_marker_genes if not (g in seen or seen.add(g))]

    if len(all_marker_genes) == 0:
        raise ValueError("No marker genes from marker_dict were found in adata.var_names.")

    expr = adata2[:, all_marker_genes].X
    if sparse.issparse(expr):
        expr = expr.toarray()

    expr_df = pd.DataFrame(expr, index=adata2.obs_names, columns=all_marker_genes)
    expr_df[groupby_col] = adata2.obs[groupby_col].astype(str).values

    avg_expr = expr_df.groupby(groupby_col).mean()

    top_marker_dict = OrderedDict()

    for ct in final_order:
        if ct not in marker_dict_present:
            continue
        if ct not in avg_expr.index:
            continue

        genes_here = [g for g in marker_dict_present[ct] if g in avg_expr.columns]
        if len(genes_here) == 0:
            continue

        top_genes = (
            avg_expr.loc[ct, genes_here]
            .sort_values(ascending=False)
            .head(top_n)
            .index
            .tolist()
        )
        top_marker_dict[ct] = top_genes

    return adata2, top_marker_dict, avg_expr, marker_dict_present, missing_marker_summary


def plot_top_marker_dotplot_and_heatmap(
    adata,
    marker_dict,
    groupby_col="label_consensus",
    top_n=3,
    prefix="dataset",
    use_raw=False,
):
    """
    Main wrapper:
    1. select top_n highly expressed marker genes per cluster
    2. plot dotplot
    3. plot matrixplot
    """
    adata2, top_marker_dict, avg_expr, marker_dict_present, missing_marker_summary = get_top_n_marker_genes_by_cluster_mean(
        adata,
        marker_dict=marker_dict,
        groupby_col=groupby_col,
        top_n=top_n,
    )

    print("Top marker genes selected:")
    for k, v in top_marker_dict.items():
        print(f"{k}: {v}")

    print("\nMissing marker genes:")
    for k, v in missing_marker_summary.items():
        print(f"{k}: {v}")

    # -------------------------
    # Dot plot
    # -------------------------
    sc.pl.dotplot(
        adata2,
        var_names=top_marker_dict,
        groupby=groupby_col,
        standard_scale="var",
        use_raw=use_raw,
        dendrogram=False,
        figsize=(14, 7),
        color_map="Blues",
        dot_max=0.7,
        show=False
    )
    plt.savefig(f"{prefix}_top{top_n}_marker_dotplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # Matrix plot / heatmap
    # -------------------------
    sc.pl.matrixplot(
        adata2,
        var_names=top_marker_dict,
        groupby=groupby_col,
        standard_scale="var",
        use_raw=use_raw,
        dendrogram=False,
        cmap="viridis",
        figsize=(14, 7),
        show=False
    )
    plt.savefig(f"{prefix}_top{top_n}_marker_matrixplot.png", dpi=300, bbox_inches="tight")
    plt.show()

    return adata2, top_marker_dict, avg_expr

In [ ]:
MARKER_DICT = SKIN_MARKERS.copy()   # your PBMC marker dictionary
groupby_col = "label_consensus"

adata_plot, top3_marker_dict, avg_expr = plot_top_marker_dotplot_and_heatmap(
    adata,
    marker_dict=MARKER_DICT,
    groupby_col=groupby_col,
    top_n=3,
    prefix="pbmc_label_consensus",
    use_raw=False
)

In [ ]:


import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, accuracy_score, confusion_matrix

def compare_labels_ari_acc_spec_sens(
    adata,
    pred_col,
    ref_col,
    ignore_labels=("unlabeled", "Unknown"),
    evaluate_on="both"   # "all" or "labeled" or "both"
):
    """
    Compare adata.obs[pred_col] (prediction) vs adata.obs[ref_col] (reference)
    using:
      - ARI
      - Accuracy
      - Specificity (macro, one-vs-rest)
      - Sensitivity/Recall (macro, one-vs-rest)
    """

    y_pred_all = adata.obs[pred_col].astype(str).values
    y_ref_all  = adata.obs[ref_col].astype(str).values

    ignore = set(map(str, ignore_labels))
    pred_labeled = ~np.isin(y_pred_all, list(ignore))
    ref_labeled  = ~np.isin(y_ref_all,  list(ignore))
    mask_labeled = pred_labeled & ref_labeled

    def _spec_sens_macro(yr, yp):
        """
        Returns (specificity_macro, sensitivity_macro) for multiclass via one-vs-rest.
        """
        classes = np.unique(np.concatenate([yr, yp]))
        specs, sens = [], []

        for cls in classes:
            yr_bin = (yr == cls).astype(int)
            yp_bin = (yp == cls).astype(int)

            cm = confusion_matrix(yr_bin, yp_bin, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()

            spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan     # TNR
            sen  = tp / (tp + fn) if (tp + fn) > 0 else np.nan     # TPR (Recall)

            specs.append(spec)
            sens.append(sen)

        return float(np.nanmean(specs)), float(np.nanmean(sens))

    def _metrics(mask):
        yr = y_ref_all[mask]
        yp = y_pred_all[mask]
        if yp.size == 0:
            return {
                "ARI": np.nan,
                "Accuracy": np.nan,
                "Specificity_macro": np.nan,
                "Sensitivity_macro": np.nan,
                "n_eval": 0
            }

        spec_m, sens_m = _spec_sens_macro(yr, yp)

        return {
            "ARI": float(adjusted_rand_score(yr, yp)),
            "Accuracy": float(accuracy_score(yr, yp)),
            "Specificity_macro": float(spec_m),
            "Sensitivity_macro": float(sens_m),
            "n_eval": int(yp.size),
        }

    out = {
        "pred_col": pred_col,
        "ref_col": ref_col,
        "coverage_pred": float(pred_labeled.mean()),
        "coverage_ref": float(ref_labeled.mean()),
        "coverage_both": float(mask_labeled.mean()),
        "n_total": int(len(y_pred_all)),
    }

    if evaluate_on in ("all", "both"):
        out_all = _metrics(np.ones(len(y_pred_all), dtype=bool))
        out.update({f"all_{k}": v for k, v in out_all.items()})

    if evaluate_on in ("labeled", "both"):
        out_lab = _metrics(mask_labeled)
        out.update({f"labeled_{k}": v for k, v in out_lab.items()})

    return out

In [ ]:
pred_cols = [
    "label_consensus",   # ✅ added Leiden-based mapped labels
    "label_knn",
    "label_rf",
    "label_kmeans",
    "label_svm_rbf",
    "label_mlp",
]

ref_col = "cell_type_true"

rows = []
for c in pred_cols:
    if c in adata.obs:
        rows.append(compare_labels_ari_acc_spec_sens(adata, c, ref_col, evaluate_on="both"))

df_metrics = pd.DataFrame(rows)

cols_show = [
    "pred_col",
    "coverage_pred",
    "coverage_both",
    "labeled_ARI",
    "labeled_Accuracy",
    "labeled_Specificity_macro",
    "labeled_Sensitivity_macro",
    "labeled_n_eval",
]

print(df_metrics[cols_show].sort_values("labeled_ARI", ascending=False))

df_metrics.to_csv("classwise_metrics.csv", index=False)
print("Saved: classwise_metrics.csv")

In [ ]:
if RUN_COMPARISONS:
    #code for scanvi
    import numpy as np
    import pandas as pd
    import scanpy as sc
    import scvi

    # -----------------------------------
    # 1. Copy data
    # -----------------------------------
    adata_scanvi = adata.copy()

    # -----------------------------------
    # 2. Make sure raw counts exist
    # scVI/scANVI should use counts, not log-normalized X
    # -----------------------------------
    if "counts" not in adata_scanvi.layers:
        # if your current X is counts, store it
        adata_scanvi.layers["counts"] = adata_scanvi.X.copy()

    # -----------------------------------
    # 3. Choose the true / reference label column
    # change this if needed
    # -----------------------------------
    label_col = "pseudo_cell_type"   # or "cell_type_true"

    adata_scanvi.obs[label_col] = adata_scanvi.obs[label_col].astype(str)

    # -----------------------------------
    # 4. Hide some labels to create unlabeled cells
    # -----------------------------------
    rng = np.random.default_rng(42)

    train_frac = 0.30   # keep only 30% labels visible
    y = adata_scanvi.obs[label_col].astype(str).values
    idx = np.arange(adata_scanvi.n_obs)

    keep_idx = []
    for ct in np.unique(y):
        ct_idx = idx[y == ct]
        n_keep = max(1, int(round(train_frac * len(ct_idx))))
        chosen = rng.choice(ct_idx, size=n_keep, replace=False)
        keep_idx.extend(chosen.tolist())

    keep_idx = np.array(sorted(set(keep_idx)))
    mask_keep = np.zeros(adata_scanvi.n_obs, dtype=bool)
    mask_keep[keep_idx] = True

    adata_scanvi.obs["label_for_scanvi"] = np.where(
        mask_keep,
        adata_scanvi.obs[label_col].astype(str),
        "unlabeled"
    )
    adata_scanvi.obs["label_for_scanvi"] = pd.Categorical(adata_scanvi.obs["label_for_scanvi"])

    print(adata_scanvi.obs["label_for_scanvi"].value_counts())

    # -----------------------------------
    # 5. Setup anndata for scVI
    # if no batch column, remove batch_key
    # -----------------------------------
    if "batch" in adata_scanvi.obs.columns:
        scvi.model.SCVI.setup_anndata(
            adata_scanvi,
            layer="counts",
            batch_key="batch",
            labels_key="label_for_scanvi"
        )
    else:
        scvi.model.SCVI.setup_anndata(
            adata_scanvi,
            layer="counts",
            labels_key="label_for_scanvi"
        )

    # -----------------------------------
    # 6. Train scVI
    # -----------------------------------
    vae = scvi.model.SCVI(
        adata_scanvi,
        n_latent=5
    )
    vae.train()

    # -----------------------------------
    # 7. Initialize and train scANVI
    # -----------------------------------
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        vae,
        adata=adata_scanvi,
        labels_key="label_for_scanvi",
        unlabeled_category="unlabeled"
    )
    scanvi_model.train(max_epochs=5)

    # -----------------------------------
    # 8. Predict labels
    # -----------------------------------
    adata_scanvi.obs["label_scanvi"] = scanvi_model.predict(adata_scanvi)
    adata_scanvi.obs["scanvi_conf"] = scanvi_model.predict(adata_scanvi, soft=True).max(axis=1)

    #print(adata_scanvi.obs["label_scanvi"].value_counts())

    #adata_scanvi.obsm["X_scanvi"] = scanvi_model.get_latent_representation(adata_scanvi)
    #sc.pp.neighbors(adata_scanvi, use_rep="X_scanvi")
    #sc.tl.umap(adata_scanvi)

    # -----------------------------------
    # 10. Plot
    # -----------------------------------
    #sc.pl.umap(adata_scanvi, color=[label_col, "label_for_scanvi", "label_scanvi"],wspace=0.4)
    adata.obs["label_scanvi"] = adata_scanvi.obs["label_scanvi"].astype(str).values

In [ ]:
if RUN_COMPARISONS:
    # The preceding comparison cell computes scANVI labels in memory.
    # Optional import from another run requires author-supplied, barcode-indexed predictions.
    if USE_EXTERNAL_PBMC_LABELS:
        pred = pd.read_csv(PREPARED / "label_scanvi.csv", dtype=str)
        pred = pred.set_index("barcode", verify_integrity=True).reindex(adata.obs_names)
        if pred["label_scanvi"].isna().any():
            raise ValueError("Missing external prediction for one or more PBMC barcodes.")
        adata.obs["label_scanvi"] = pred["label_scanvi"].to_numpy()


In [ ]:
if RUN_COMPARISONS:
    # Optional historical label substitution. Confirm its biological mapping before use.
    if USE_LEGACY_LABEL_MAP:
        adata.obs["label_scanvi"] = adata.obs["label_scanvi"].astype(str).replace("CD4+ T Helper2", "CD34+")

In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd
    import scanpy as sc
    import matplotlib.pyplot as plt

    # -----------------------------
    # 1. Choose object and columns
    # -----------------------------
    adata_umap = adata.copy()   # safer than adata_umap = adata

    truth_col = "pseudo_cell_type"        # ground truth label
    consensus_col = "label_consensus"   # scAICME
    scanvi_col = "label_scanvi"         # scANVI
    leiden_col = "label_leiden"         # Leiden clusters

    rare_frac = 0.01   # < 1%

    # -----------------------------
    # 2. Make sure UMAP exists
    # -----------------------------
    if "X_umap" not in adata_umap.obsm:
        if "X_pca" not in adata_umap.obsm:
            sc.pp.pca(adata_umap, n_comps=30)
        if "connectivities" not in adata_umap.obsp:
            sc.pp.neighbors(adata_umap, n_neighbors=15, n_pcs=30)
        sc.tl.umap(adata_umap)

    # -----------------------------
    # 3. Identify rare cell types from truth
    # -----------------------------
    N = adata_umap.n_obs
    cell_counts = adata_umap.obs[truth_col].astype(str).value_counts()
    rare_types = cell_counts[cell_counts < rare_frac * N].index.tolist()
    rare_set = set(rare_types)

    print("Rare cell types (freq < 1%):")
    print(rare_types)

    # -----------------------------
    # 4. Ground-truth rare highlight
    # -----------------------------
    adata_umap.obs["rare_true_highlight"] = np.where(
        adata_umap.obs[truth_col].astype(str).isin(rare_set),
        adata_umap.obs[truth_col].astype(str),
        "Other"
    )

    # -----------------------------
    # 5. Rare highlight for direct label columns
    #    (consensus and scanvi already contain cell-type names)
    # -----------------------------
    def make_rare_highlight_direct(adata_obj, pred_col, out_col, rare_set):
        pred = adata_obj.obs[pred_col].astype(str)
        adata_obj.obs[out_col] = np.where(pred.isin(rare_set), pred, "Other")

    make_rare_highlight_direct(
        adata_umap, consensus_col, "rare_pred_consensus", rare_set
    )

    make_rare_highlight_direct(
        adata_umap, scanvi_col, "rare_pred_scanvi", rare_set
    )

    # -----------------------------
    # 6. Rare highlight for Leiden
    #    Leiden cluster IDs must be mapped to truth labels first
    # -----------------------------
    cluster_to_truth = (
        pd.crosstab(
            adata_umap.obs[leiden_col].astype(str),
            adata_umap.obs[truth_col].astype(str)
        )
        .idxmax(axis=1)
        .to_dict()
    )

    adata_umap.obs["label_leiden_mapped"] = (
        adata_umap.obs[leiden_col].astype(str).map(cluster_to_truth).fillna("Other")
    )

    adata_umap.obs["rare_pred_leiden"] = np.where(
        adata_umap.obs["label_leiden_mapped"].astype(str).isin(rare_set),
        adata_umap.obs["label_leiden_mapped"].astype(str),
        "Other"
    )

    print("\nLeiden cluster -> mapped truth label:")
    print(cluster_to_truth)

    # -----------------------------
    # 7. Plot
    # -----------------------------
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # (A) Ground truth full labels
    sc.pl.umap(
        adata_umap,
        color=truth_col,
        ax=axes[0, 0],
        show=False,
        size=8,
        title="Ground truth cell types",
        legend_loc="right margin",
        frameon=False
    )

    # (B) Rare cells recovered by scAICME
    sc.pl.umap(
        adata_umap,
        color="rare_pred_consensus",
        ax=axes[0, 1],
        show=False,
        size=10,
        title="Rare cell types recovered (scAICME)",
        legend_loc="right margin",
        frameon=False
    )

    # (C) Rare cells recovered by scANVI
    sc.pl.umap(
        adata_umap,
        color="rare_pred_scanvi",
        ax=axes[1, 0],
        show=False,
        size=10,
        title="Rare cell types recovered (scANVI)",
        legend_loc="right margin",
        frameon=False
    )

    # (D) Rare cells recovered by Leiden
    sc.pl.umap(
        adata_umap,
        color="rare_pred_leiden",
        ax=axes[1, 1],
        show=False,
        size=10,
        title="Rare cell types recovered (Seurat)",
        legend_loc="right margin",
        frameon=False
    )

    plt.tight_layout()
    plt.savefig("combined_umap_rare_analysis_all_methods.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

def clustering_metrics(adata, cluster_col, X_key="X_pca"):
    X = np.asarray(adata.obsm[X_key])
    labels = adata.obs[cluster_col].astype(str).values

    if len(np.unique(labels)) < 2:
        return {
            "method": cluster_col,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan
        }

    return {
        "method": cluster_col,
        "silhouette": silhouette_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels)
    }

# only scANVI label
rows = []

if "label_consensus" in adata.obs:
    rows.append(clustering_metrics(adata, "label_consensus"))

df_cluster_metrics = pd.DataFrame(rows)

print(df_cluster_metrics)

df_cluster_metrics.to_csv("pbmc_scACIME_cluster_metrics.csv", index=False)

#print("Saved: pbmc_scanvi_cluster_metrics.csv")